In [ ]:
!pip install -q datasets sentence-transformers rank-bm25

import time
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo ativo: {DEVICE.upper()}")

Dispositivo ativo: CUDA


In [ ]:
import requests
from huggingface_hub import HfApi, hf_hub_url

NUM_QUERIES = 300       # Quantidade de perguntas únicas avaliadas
NUM_DISTRACTORS = 5000  # Passagens falsas extras para compor a base de busca
POOL_SIZE = 250000      # Quantidade inicial de passagens lidas via streaming (~35 MB)

def load_mmarco_subsample(num_queries, num_distractors, pool_size=250000):
    print("Localizando arquivos oficiais no repositório unicamp-dl/mmarco...")
    api = HfApi()
    repo_files = api.list_repo_files("unicamp-dl/mmarco", repo_type="dataset")

    # Localiza automaticamente os caminhos exatos dos arquivos em português no repositório
    col_path = [f for f in repo_files if f.endswith("portuguese_collection.tsv")][0]
    qry_path = [f for f in repo_files if f.endswith("portuguese_queries.train.tsv")][0]
    tri_path = [f for f in repo_files if f.endswith("triples.train.ids.small.tsv")][0]

    col_url = hf_hub_url("unicamp-dl/mmarco", col_path, repo_type="dataset")
    qry_url = hf_hub_url("unicamp-dl/mmarco", qry_path, repo_type="dataset")
    tri_url = hf_hub_url("unicamp-dl/mmarco", tri_path, repo_type="dataset")

    # 1. Lê um pool inicial de passagens em português via streaming (sem baixar os 8.8 milhões)
    print(f"1/3 - Coletando pool de {pool_size} passagens em português via streaming...")
    passage_pool = {}
    with requests.get(col_url, stream=True) as r:
        r.raise_for_status()
        for raw_line in r.iter_lines():
            if not raw_line:
                continue
            parts = raw_line.decode("utf-8", errors="ignore").split("\t", 1)
            if len(parts) == 2:
                passage_pool[parts[0]] = parts[1]
            if len(passage_pool) >= pool_size:
                break

    # 2. Carrega o dicionário de perguntas de treino em português
    print("2/3 - Carregando perguntas em português...")
    queries_pool = {}
    with requests.get(qry_url, stream=True) as r:
        r.raise_for_status()
        for raw_line in r.iter_lines():
            if not raw_line:
                continue
            parts = raw_line.decode("utf-8", errors="ignore").split("\t", 1)
            if len(parts) == 2:
                queries_pool[parts[0]] = parts[1]

    # 3. Cruza as triplas (query_id, pos_id, neg_id) para montar o Ground Truth (qrels)
    print("3/3 - Montando subamostra anotada (Ground Truth + Distratores)...")
    queries = []
    corpus_dict = {}  # texto -> doc_id
    corpus = []       # Lista final de textos de passagens
    qrels = {}        # Ground Truth: {query_id_local: {doc_id_relevante}}
    seen_qids = set() # Evita perguntas duplicadas

    def add_passage(text):
        if text not in corpus_dict:
            doc_id = len(corpus)
            corpus_dict[text] = doc_id
            corpus.append(text)
        return corpus_dict[text]

    with requests.get(tri_url, stream=True) as r:
        r.raise_for_status()
        for raw_line in r.iter_lines():
            if not raw_line:
                continue
            parts = raw_line.decode("utf-8", errors="ignore").split("\t")
            if len(parts) != 3:
                continue
            qid, pos_id, neg_id = parts[0], parts[1], parts[2]

            # Verifica se a pergunta é inédita e se a resposta correta está no nosso pool
            if qid not in seen_qids and qid in queries_pool and pos_id in passage_pool:
                seen_qids.add(qid)
                local_q_id = len(queries)
                queries.append(queries_pool[qid])

                pos_doc_id = add_passage(passage_pool[pos_id])
                qrels[local_q_id] = {pos_doc_id}

                if neg_id in passage_pool:
                    add_passage(passage_pool[neg_id])

                if len(queries) >= num_queries:
                    break

    # Preenche o restante do corpus com passagens negativas/distratoras do pool
    target_corpus_size = len(corpus) + num_distractors
    for pid, p_text in passage_pool.items():
        add_passage(p_text)
        if len(corpus) >= target_corpus_size:
            break

    print(f"\nSubamostra pronta: {len(queries)} queries únicas | {len(corpus)} passagens únicas no corpus.")
    return queries, corpus, qrels

queries, corpus, qrels = load_mmarco_subsample(NUM_QUERIES, NUM_DISTRACTORS, POOL_SIZE)

Localizando arquivos oficiais no repositório unicamp-dl/mmarco...
1/3 - Coletando pool de 250000 passagens em português via streaming...
2/3 - Carregando perguntas em português...
3/3 - Montando subamostra anotada (Ground Truth + Distratores)...

Subamostra pronta: 300 queries únicas | 5312 passagens únicas no corpus.


In [ ]:
MAX_K = 100

def tokenize_pt(text):
    """Tokenizador compartilhado: extrai palavras alfanuméricas em minúsculas."""
    return re.findall(r'\w+', text.lower())

# ==============================================================================
# 1A. RECUPERAÇÃO COM TF-IDF + SIMILARIDADE DE COSSENO
# ==============================================================================
print("1. Construindo matriz TF-IDF do corpus...")
vectorizer = TfidfVectorizer(tokenizer=tokenize_pt, lowercase=True, token_pattern=None)
tfidf_corpus_matrix = vectorizer.fit_transform(corpus)

tfidf_rankings = {}
tfidf_latencies = []

print(f"Recuperando Top-{MAX_K} com TF-IDF + Cosseno...")
for q_id, q_text in enumerate(tqdm(queries, desc="TF-IDF")):
    t0 = time.perf_counter()
    query_vec = vectorizer.transform([q_text])
    scores = cosine_similarity(query_vec, tfidf_corpus_matrix).flatten()
    top_indices = np.argsort(scores)[::-1][:MAX_K].tolist()

    tfidf_latencies.append((time.perf_counter() - t0) * 1000) # ms
    tfidf_rankings[q_id] = top_indices

# ==============================================================================
# 1B. RECUPERAÇÃO COM BM25 OKAPI
# ==============================================================================
print("\n2. Tokenizando corpus e construindo índice BM25...")
tokenized_corpus = [tokenize_pt(doc) for doc in tqdm(corpus, desc="Tokenizando BM25")]
bm25 = BM25Okapi(tokenized_corpus)

bm25_rankings = {}
bm25_latencies = []

print(f"Recuperando Top-{MAX_K} com BM25...")
for q_id, q_text in enumerate(tqdm(queries, desc="BM25")):
    t0 = time.perf_counter()
    tokenized_query = tokenize_pt(q_text)
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:MAX_K].tolist()

    bm25_latencies.append((time.perf_counter() - t0) * 1000) # ms
    bm25_rankings[q_id] = top_indices

print("\nRecuperação de 1º estágio (TF-IDF e BM25) concluída com sucesso!")

1. Construindo matriz TF-IDF do corpus...
Recuperando Top-100 com TF-IDF + Cosseno...


TF-IDF:   0%|          | 0/300 [00:00<?, ?it/s]


2. Tokenizando corpus e construindo índice BM25...


Tokenizando BM25:   0%|          | 0/5312 [00:00<?, ?it/s]

Recuperando Top-100 com BM25...


BM25:   0%|          | 0/300 [00:00<?, ?it/s]


Recuperação de 1º estágio (TF-IDF e BM25) concluída com sucesso!


In [ ]:
def compute_metrics(ranked_doc_ids, relevant_doc_ids, k=10):
    top_k = ranked_doc_ids[:k]

    # 1. MRR@10
    mrr = 0.0
    for rank, doc_id in enumerate(top_k, start=1):
        if doc_id in relevant_doc_ids:
            mrr = 1.0 / rank
            break

    # 2. nDCG@10
    dcg = 0.0
    for rank, doc_id in enumerate(top_k, start=1):
        if doc_id in relevant_doc_ids:
            dcg += 1.0 / np.log2(rank + 1)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, min(len(relevant_doc_ids), k) + 1))
    ndcg = (dcg / idcg) if idcg > 0 else 0.0

    # 3. Recall@10
    hits = sum(1 for doc_id in top_k if doc_id in relevant_doc_ids)
    recall = hits / len(relevant_doc_ids) if relevant_doc_ids else 0.0

    return mrr, ndcg, recall

def find_rank(ranked_doc_ids, relevant_doc_ids):
    """Retorna a posição exata (1-indexed) do documento relevante na lista."""
    for rank, doc_id in enumerate(ranked_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            return rank
    return -1

In [ ]:
MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
print(f"Carregando Cross-Encoder: {MODEL_NAME}...")
cross_encoder = CrossEncoder(MODEL_NAME, device=DEVICE, max_length=512)

K_VALUES = [10, 25, 50, 100]
results_summary = []

# Função que avalia uma baseline léxica (TF-IDF ou BM25) e aplica o Cross-Encoder sobre ela
def evaluate_pipeline(stage1_name, stage1_rankings, stage1_latencies):
    # 1. Métricas da Baseline Pura (1º Estágio isolado)
    base_metrics = [compute_metrics(stage1_rankings[q_id], qrels[q_id], k=10) for q_id in range(len(queries))]
    results_summary.append({
        "Pipeline": f"{stage1_name} (Puro - Baseline)",
        "1o_Estagio": stage1_name,
        "Top_K_Rerank": "-",
        "MRR@10": round(np.mean([m[0] for m in base_metrics]), 4),
        "nDCG@10": round(np.mean([m[1] for m in base_metrics]), 4),
        "Recall@10": round(np.mean([m[2] for m in base_metrics]), 4),
        "Latencia_Media_ms": round(np.mean(stage1_latencies), 2)
    })

    # 2. Reranqueamento com Cross-Encoder para cada valor de K
    reranked_top100_dict = {}

    for k in K_VALUES:
        ce_mrr, ce_ndcg, ce_rec, ce_latencies = [], [], [], []

        for q_id, q_text in enumerate(tqdm(queries, desc=f"{stage1_name} -> Cross-Encoder (Top-{k})")):
            candidate_ids = stage1_rankings[q_id][:k]
            pairs = [[q_text, corpus[doc_id]] for doc_id in candidate_ids]

            t0 = time.perf_counter()
            ce_scores = cross_encoder.predict(pairs, batch_size=32, show_progress_bar=False)
            ce_latencies.append((time.perf_counter() - t0) * 1000)

            reranked_order = np.argsort(ce_scores)[::-1]
            reranked_doc_ids = [candidate_ids[idx] for idx in reranked_order]

            m, n, r = compute_metrics(reranked_doc_ids, qrels[q_id], k=10)
            ce_mrr.append(m)
            ce_ndcg.append(n)
            ce_rec.append(r)

            if k == MAX_K:
                reranked_top100_dict[q_id] = reranked_doc_ids

        results_summary.append({
            "Pipeline": f"{stage1_name} + Cross-Encoder (Top-{k})",
            "1o_Estagio": stage1_name,
            "Top_K_Rerank": k,
            "MRR@10": round(np.mean(ce_mrr), 4),
            "nDCG@10": round(np.mean(ce_ndcg), 4),
            "Recall@10": round(np.mean(ce_rec), 4),
            "Latencia_Media_ms": round(np.mean(stage1_latencies) + np.mean(ce_latencies), 2)
        })

    return reranked_top100_dict

# Executa a bateria para TF-IDF e para BM25
ce_from_tfidf = evaluate_pipeline("TF-IDF", tfidf_rankings, tfidf_latencies)
ce_from_bm25  = evaluate_pipeline("BM25", bm25_rankings, bm25_latencies)

# Exibe e salva a Tabela Principal de Resultados
df_results = pd.DataFrame(results_summary)
df_results.to_csv("resultados_comparativos_tfidf_bm25_ce.csv", index=False)

print("\n================ TABELA COMPARATIVA FINAL ================")
display(df_results)

Carregando Cross-Encoder: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1...


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

... (11 outputs left out)

BM25 -> Cross-Encoder (Top-50):   0%|          | 0/300 [00:00<?, ?it/s]

BM25 -> Cross-Encoder (Top-100):   0%|          | 0/300 [00:00<?, ?it/s]


================ TABELA COMPARATIVA FINAL ================


,Pipeline,1o_Estagio,Top_K_Rerank,MRR@10,nDCG@10,Recall@10,Latencia_Media_ms
0,TF-IDF (Puro - Baseline),TF-IDF,-,0.7554,0.7865,0.8833,4.24
1,TF-IDF + Cross-Encoder (Top-10),TF-IDF,10,0.8672,0.8714,0.8833,40.61
2,TF-IDF + Cross-Encoder (Top-25),TF-IDF,25,0.8940,0.9005,0.9200,106.00
3,TF-IDF + Cross-Encoder (Top-50),TF-IDF,50,0.9050,0.9128,0.9367,167.48
4,TF-IDF + Cross-Encoder (Top-100),TF-IDF,100,0.9094,0.9170,0.9400,292.76
5,BM25 (Puro - Baseline),BM25,-,0.8008,0.8221,0.8900,10.16
6,BM25 + Cross-Encoder (Top-10),BM25,10,0.8724,0.8769,0.8900,47.18
7,BM25 + Cross-Encoder (Top-25),BM25,25,0.8949,0.9020,0.9233,107.75
8,BM25 + Cross-Encoder (Top-50),BM25,50,0.9002,0.9074,0.9300,182.83
9,BM25 + Cross-Encoder (Top-100),BM25,100,0.9130,0.9196,0.9400,309.34


In [ ]:
qualitative_records = []

for q_id, q_text in enumerate(queries):
    rel_doc_id = list(qrels[q_id])[0]

    tfidf_pos = find_rank(tfidf_rankings[q_id], qrels[q_id])
    bm25_pos  = find_rank(bm25_rankings[q_id], qrels[q_id])
    ce_tfidf_pos = find_rank(ce_from_tfidf[q_id], qrels[q_id])
    ce_bm25_pos  = find_rank(ce_from_bm25[q_id], qrels[q_id])

    qualitative_records.append({
        "query_id": q_id,
        "query_text": q_text,
        "query_word_count": len(q_text.split()),
        "relevant_passage": corpus[rel_doc_id],
        "top1_tfidf_passage": corpus[tfidf_rankings[q_id][0]],
        "top1_bm25_passage": corpus[bm25_rankings[q_id][0]],
        "tfidf_rank": tfidf_pos,
        "bm25_rank": bm25_pos,
        "ce_over_tfidf_rank": ce_tfidf_pos,
        "ce_over_bm25_rank": ce_bm25_pos
    })

df_qual = pd.DataFrame(qualitative_records)
df_qual.to_csv("analise_qualitativa_completa.csv", index=False)

# Filtra os casos em que TANTO o TF-IDF quanto o BM25 erraram (rank > 10), mas o Cross-Encoder cravou 1º lugar!
ambos_falharam_ce_acertou = df_qual[
    (df_qual["tfidf_rank"] > 10) &
    (df_qual["bm25_rank"] > 10) &
    (df_qual["ce_over_bm25_rank"] == 1)
].sort_values(by="bm25_rank", ascending=False)

ambos_falharam_ce_acertou.to_csv("casos_sucesso_absoluto_ce.csv", index=False)

print(f"Total de queries em que AMBOS (TF-IDF e BM25) ficaram fora do Top-10 e o Cross-Encoder subiu para 1º lugar: {len(ambos_falharam_ce_acertou)}\n")

# Imprime os 3 melhores exemplos para colocar no artigo
for idx, row in ambos_falharam_ce_acertou.head(3).iterrows():
    print("="*90)
    print(f"PERGUNTA (ID {row['query_id']}): {row['query_text']}")
    print(f"Posições -> TF-IDF: {row['tfidf_rank']}º | BM25: {row['bm25_rank']}º  ===>  Cross-Encoder: 1º LUGAR!")
    print(f"\n[Falso Positivo Léxico (1º lugar no BM25)]:\n{row['top1_bm25_passage']}")
    print(f"\n[Passagem Verdadeira Resgatada pelo Cross-Encoder]:\n{row['relevant_passage']}\n")

Total de queries em que AMBOS (TF-IDF e BM25) ficaram fora do Top-10 e o Cross-Encoder subiu para 1º lugar: 11

PERGUNTA (ID 69): Crowne Plaza faz parte da rede de hotéis
Posições -> TF-IDF: 80º | BM25: 67º  ===>  Cross-Encoder: 1º LUGAR!

[Falso Positivo Léxico (1º lugar no BM25)]:
A cidade faz parte da Área Micropolitana de Marion-Herrin e faz parte da Área Estatística Combinada de Carbondale-Marion-Herrin, Illinois, com 123.272 residentes, a sexta área estatística combinada mais populosa de Illinois.

[Passagem Verdadeira Resgatada pelo Cross-Encoder]:
InterContinental, Holiday InnÃƒâ € šÃ‚Â® Hotels & Resorts, Holiday Inn Club VacationsÃƒâ € šÃ‚Â®, Holiday Inn ExpressÃƒâ € šÃ‚Â® Hotels, Crowne PlazaÃƒâ € šÃ‚Â® Hotels & Resorts, Hotel IndigoÃƒâ € šÃ‚Â ®, HUALUXEÃƒÂ ¢ Ã‚â € žÃ‚Â ¢ Hotels e Resorts, EVENÃƒÂ ¢ Ã‚â € žÃ‚Â ¢ Hotels, Candlewood SuitesÃƒâ € šÃ‚Â®, Staybridge SuitesÃƒâ € šÃ‚Â® e IHGÃƒâ € šÃ‚Â® Rewards Club.

PERGUNTA (ID 263): o que é saúde definitiva
Posições -> TF-IDF: 36º